In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
import joblib 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def execute_final_training_pipeline():
    
    df = pd.read_csv("train.csv")
    print(f"here is before data columns :  {list(df.columns)}")
    
    df['TotalSF'] = df['GrLivArea'] + df.get('TotalBsmt', 0)
    df['TotalBath'] = df.get('Fullbath', 0) + 0.5*df.get('HalfBath', 0) + df.get('BsmtFullBath', 0)
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    
    print (f"now this  the after the adding new fatures:  {list(df.columns)}")
    
    NUM_FEATURES = ['TotalSF', 'LotArea', 'HouseAge', 'GarageCars', 'Fireplaces', 'TotalBath', 'IsRemodeled']
    CAT_FEATURES = ['Neighborhood', 'SaleCondition']
    
    FEATURES = NUM_FEATURES + CAT_FEATURES
    TARGET = 'SalePrice'
    
    
    
    
    
    X = df[FEATURES]
    y_raw = df[TARGET]
    y_log = np.log1p(y_raw)
    
    
    X_train, X_val, y_train_log, y_val_log = train_test_split(X, y_log, test_size= 0.2, random_state = 42, shuffle= True)
    
    numerical_pipeline = Pipeline(steps = [
        ('imputer', SimpleImputer(strategy= 'median')),
        ('scaler', PowerTransformer(method = 'yeo-johnson'))
    ])
    
    categorical_pipeline = Pipeline(steps = [
        ('imputer' , SimpleImputer(strategy= 'most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown= 'ignore', sparse_output= False))
    ])
    
    preprocessor = ColumnTransformer(transformers= [
        ('num_transform', numerical_pipeline, NUM_FEATURES),
        ('cat_transform', categorical_pipeline, CAT_FEATURES)
    ])
    
    base_pipeline = Pipeline(steps = [
        ('processor', preprocessor),
        ('regressor' , GradientBoostingRegressor(random_state= 42))
    ])
    
    param_grid = [{
        'regressor__n_estimators': [100, 200, 300],
        'regressor__learning_rate': [0.03, 0.05, 0.1],
        'regressor__max_depth': [3, 4, 5],
        'regressor__min_samples_leaf': [2, 4, 6],
        'regressor__min_samples_split': [5, 10],
        'regressor__max_features': ['sqrt', 0.8],
        'regressor__subsample': [0.8, 0.9]
    }]
    
    grid_search = GridSearchCV(base_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
    grid_search.fit(X_train, y_train_log)
    best_pipeline = grid_search.best_estimator_
    print(f"Best Hyperparameters Identified: {grid_search.best_params_}")
    
    
##################################################################################
# ── STEP 7: EVALUATE INDEPENDENT HOLDOUT SCORES ───────────
    val_preds_log = best_pipeline.predict(X_val)
    val_preds_raw = np.expm1(val_preds_log)
    y_val_raw = np.expm1(y_val_log)

    print("\n📈 PRODUCTION HOLDOUT TESTING METRICS (SHUFFLED) 📈")
    print(f"• Model Variance Explained (R² Score): {r2_score(y_val_raw, val_preds_raw):.4f}")
    print(f"• Mean Absolute Error (MAE):           ${mean_absolute_error(y_val_raw, val_preds_raw):,.2f}")
    print(f"• Root Mean Squared Error (RMSE):       ${np.sqrt(mean_squared_error(y_val_raw, val_preds_raw)):,.2f}")
    print("-" * 60)

    # ── STEP 8: VISUALIZING DISTRIBUTIONS AND OUTLIERS ──────────
    print("🎨 [TRAIN] Generating full mathematical plots dashboard...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.patch.set_facecolor('#0f1117')
    
    for ax in axes:
        ax.set_facecolor('#1a1d27')
        ax.tick_params(colors='#aaa', labelsize=10)
        ax.xaxis.label.set_color('#aaa')
        ax.yaxis.label.set_color('#aaa')

    # Plot A: Target Distribution Transformation Check (Bell Curve Conversion)
    sns.kdeplot(y_raw, color='#D85A30', fill=True, alpha=0.4, ax=axes[0], label="Raw Target (Skewed)")
    ax_twin = axes[0].twiny()
    sns.kdeplot(y_log, color='#1D9E75', fill=True, alpha=0.3, ax=ax_twin, label="Log Transformed")
    ax_twin.tick_params(colors='#aaa')
    axes[0].set_title('Target Transformation Analysis (Raw Skew vs. Normal Bell Curve)', color='white', fontweight='bold', pad=15)
    axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
    axes[0].set_xlabel("Raw SalePrice Scale ($)")
    ax_twin.set_xlabel("Compressed Logarithm Scale")

    # Plot B: Outlier Isolation Matrix via Interquartile Range Boundaries
    Q1 = df['TotalSF'].quantile(0.25)
    Q3 = df['TotalSF'].quantile(0.75)
    IQR = Q3 - Q1
    upper_fence = Q3 + 1.5 * IQR
    
    sns.boxplot(data=df, x='TotalSF', color='#7F77DD', ax=axes[1], flierprops={"markerfacecolor":"#D85A30", "markersize":6})
    axes[1].axvline(upper_fence, color='#D85A30', linestyle='--', linewidth=2, label=f"IQR Upper Boundary Fence (${upper_fence:,.0f} SF)")
    axes[1].set_title('Outliers Detection Matrix on Engineered Features (TotalSF)', color='white', fontweight='bold', pad=15)
    axes[1].set_xlabel('Total Spatial Footprint Area (sq ft)')
    axes[1].legend(facecolor='#1a1d27', labelcolor='white')

    plt.tight_layout()
    plt.savefig('training_plots_dashboard.png', facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    print("📊 Plots saved successfully as 'training_plots_dashboard.png'")

    # ── STEP 9: SERIALIZE RUNTIME ARTIFACTS ───────────────────
    print("💾 [TRAIN] Exporting operational weights to storage...")
    joblib.dump(best_pipeline, 'house_model_pipeline.pkl')
    joblib.dump(list(df['Neighborhood'].unique()), 'neighborhoods.pkl')
    joblib.dump(list(df['SaleCondition'].unique()), 'sale_conditions.pkl')
    print("🏁 Training pipeline sequence complete.\n")

if __name__ == "__main__":
    execute_final_training_pipeline()

here is before data columns :  ['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'Scr